In [1]:
%pip install openai pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import time
import pandas as pd
from openai import OpenAI
import re

# Place your OpenRouter API key in a file called 'API_key' in this directory
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="REDACTED-SEE-.env-AT-REPO-ROOT",
)

In [3]:
# DIALECT-COPA datasets
# Place the corresponding *-test.jsonl files in the datasets/ folder
# e.g. datasets/copa-en-test.jsonl, datasets/copa-sl-test.jsonl, etc.
tests = [
    #"copa-en",
    #"copa-sl",
    #"copa-hr",
    #"copa-hr-ckm",
    #"copa-mk",
    #"copa-sl-cer",
    #"copa-sr",
    #"copa-sr-tor",
    "copa-sl-prl"
]

# OpenRouter models
models = [
    #"google/gemini-3.1-pro-preview",
    #"anthropic/claude-opus-4.6",
    #"google/gemini-3.1-flash-lite-preview",
    #"anthropic/claude-sonnet-4.6",
    #"openai/gpt-5",
    #"google/gemini-2.5-pro",
    #"google/gemini-2.5-flash",
    #"openai/gpt-4o",
    #"anthropic/claude-haiku-4.5",
    #"mistralai/mistral-medium-3.1",
    #"meta-llama/llama-3.3-70b-instruct",
    #"google/gemma-4-31b-it",
    #"google/gemma-4-26b-a4b-it",
    "google/gemma-3-27b-it",
    #"qwen/qwen3-32b",
    #"openai/gpt-3.5-turbo",
    #"openai/gpt-5.4-pro",
    #"openai/gpt-5.4",
    #"mistralai/mistral-large-2512",
    #"mistralai/mistral-small-2603",
    #"meta-llama/llama-4-maverick"
]

In [4]:
def get_completion(gpt_model, prompt, retries=3):
    for attempt in range(retries):
        try:
            completion = client.chat.completions.create(
                model=gpt_model,
                response_format={"type": "json_object"},
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
            )
            choice = completion.choices[0] if completion.choices else None
            content = choice.message.content if choice else None
            if content:
                return content
            reason = getattr(choice, "finish_reason", None) if choice else None
            err = getattr(completion, "error", None)
            print(f"  empty response (finish_reason={reason}, error={err}), retry {attempt+1}/{retries}")
        except Exception as e:
            print(f"  request failed: {e}, retry {attempt+1}/{retries}")
        time.sleep(2 ** attempt)
    return None

In [5]:
def predict_gpt(df_test_name, gpt_model):
    import re
    os.makedirs("copa_submissions", exist_ok=True)

    model_name = gpt_model.split("/")[1]
    out_path = f"copa_submissions/submission-{model_name}-{df_test_name}.json"
    if os.path.exists(out_path):
        print(f"Skipping, already exists: {out_path}")
        return

    df_path = f"copa_datasets/{df_test_name}-test.jsonl"
    responses = []
    start_time = time.time()

    for line in open(df_path):
        entry = json.loads(line)

        if df_test_name == "copa-en":
            prompt = (
                'You will be given a task. The task definition is in English, '
                'as is the task itself. Here is the task!\n'
                f'Given the premise "{entry["premise"]}",'
            )
        else:
            prompt = (
                'You will be given a task. The task definition is in English, '
                'but the task itself is in another language. Here is the task!\n'
                f'Given the premise "{entry["premise"]}",'
            )

        if entry["question"] == "cause":
            prompt += " and that we are looking for the cause of this premise,"
        else:
            prompt += " and that we are looking for the result of this premise, "

        prompt += (
            f'which hypothesis is more plausible?\n'
            f'Hypothesis 1: "{entry["choice1"]}".\n'
            f'Hypothesis 2: "{entry["choice2"]}".\n\n'
            f'### Output format\n'
            f"Return a valid JSON dictionary with the following key: 'answer' "
            f"and a value should be an integer -- either 1 (if hypothesis 1 is "
            f"more plausible) or 2 (if hypothesis 2 is more plausible)."
        )

        initial_response = get_completion(gpt_model, prompt)

        if initial_response is None:
            print(f"  instance {len(responses)}: no usable output, defaulting to 0")
            responses.append(0)
            continue

        match = re.search(r'"?\'?answer"?\'?\s*:\s*([12])', initial_response)
        if match:
            responses.append(int(match.group(1)) - 1)
        else:
            print("Error extracting label:")
            print(initial_response)
            responses.append(0)

    elapsed = time.time() - start_time
    n = len(responses)
    print(f"Prediction finished. {elapsed/60:.2f} min for {n} instances — {elapsed/n:.3f} s/instance.")

    current_results = {
        "system": gpt_model,
        "predictions": [{"train": "NA (zero-shot)", "test": df_test_name, "predictions": responses}],
    }

    with open(out_path, "w") as f:
        json.dump(current_results, f)

    print(f"Saved: {out_path}")

In [6]:
# Run all models on all datasets
# Tip: comment out models or datasets you want to skip / re-run individually
for model in models:
    for test in tests:
        print(f"\n=== {model} | {test} ===")
        predict_gpt(test, model)


=== google/gemma-3-27b-it | copa-sl-prl ===
  request failed: Error code: 404 - {'error': {'message': 'All providers have been ignored. To change your default ignored providers, visit: https://openrouter.ai/settings/privacy', 'code': 404}}, retry 1/3
  request failed: Error code: 404 - {'error': {'message': 'All providers have been ignored. To change your default ignored providers, visit: https://openrouter.ai/settings/privacy', 'code': 404}}, retry 2/3
  request failed: Error code: 404 - {'error': {'message': 'All providers have been ignored. To change your default ignored providers, visit: https://openrouter.ai/settings/privacy', 'code': 404}}, retry 3/3
  instance 0: no usable output, defaulting to 0
  request failed: Error code: 404 - {'error': {'message': 'All providers have been ignored. To change your default ignored providers, visit: https://openrouter.ai/settings/privacy', 'code': 404}}, retry 1/3
  request failed: Error code: 404 - {'error': {'message': 'All providers have b

KeyboardInterrupt: 